In [ ]:
# Import core libraries for data handling and math
import numpy as np
import pandas as pd
# Import plotting library for data visualization
import matplotlib.pyplot as plt
# Import preprocessing tools
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
# Import machine learning models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
# Import evaluation metric
from sklearn.metrics import classification_report
# Import deep learning library
import tensorflow as tf



In [ ]:
# Define column names, load dataset into a DataFrame, and preview first 5 rows
cols = ['fLength', 'fWidth', 'fSize', 'fConc','fConc1' ,'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']
df = pd.read_csv("/content/magic04.data", names = cols)
df.head()

In [ ]:
# Get unique values (classes) in the target column
df['class'].unique()

In [ ]:
# Convert 'class' column to binary: 'g' -> 1, others -> 0
df['class'] = (df['class'] == 'g').astype(int)

In [ ]:
df.head()

In [ ]:
# Plot histograms for each feature comparing gamma (1) vs hadron (0)
for label in cols[:-1]:
  plt.hist(df[df["class"]==1][label], color = 'blue', label = 'gamma', alpha = 0.7, density= True)
  plt.hist(df[df["class"]==0][label], color = 'red', label = 'hydron', alpha = 0.7, density= True)
  plt.title(label)
  plt.ylabel("Probability")
  plt.xlabel(label)
  plt.legend()
  plt.show()


In [ ]:
# Shuffle dataset and split into train (60%), validation (20%), and test (20%) sets
train, vals, test = np.split(df.sample(frac = 1), [int(0.6*len(df)), int(0.8*len(df))])

In [ ]:
# Function to scale features, optionally oversample minority class, and return processed data
def scale_dataset(dataframe, oversample = False):
  X = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values
  scaler = StandardScaler()
  X = scaler.fit_transform(X)
  if oversample:
    ros = RandomOverSampler()
    X, y = ros.fit_resample(X, y)

  data = np.hstack((X,np.reshape(y,(-1,1))))
  return data, X, y

In [ ]:
print(len(train_df[train_df["class"] == 1]))#gamma
print(len(train_df[train_df["class"] == 0]))#hydron

In [ ]:
# Convert splits to DataFrames, scale features, and prepare train/validation/test sets
# Apply oversampling only to training data
train_df = pd.DataFrame(train, columns=cols)
valid_df = pd.DataFrame(vals, columns=cols)
test_df = pd.DataFrame(test, columns=cols)

train, X_train, y_train = scale_dataset(train_df, oversample= True)
valid, X_valid, y_valid = scale_dataset(valid_df, oversample= True)
test, X_test, y_test = scale_dataset(test_df, oversample= False)

In [ ]:
len(y_train)

In [ ]:
sum(y_train == 1)

In [ ]:
sum(y_train == 0)

KNN

In [ ]:
knn_model =  KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

In [ ]:
y_pred = knn_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

Naive Bayes

In [ ]:
nb_model = GaussianNB()
nb_model = nb_model.fit(X_train, y_train)

In [ ]:
y_pred = nb_model.predict(X_test)
print(classification_report(y_test, y_pred))

Logistic Regression

In [ ]:
lg_model = LogisticRegression()
lg_model = lg_model.fit(X_train, y_train)

In [ ]:
y_pred = lg_model.predict(X_test)
print(classification_report(y_test, y_pred))

SVM

In [ ]:
svm_model = SVC()
svm_model = svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

In [ ]:
y_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_pred))

Neural Network

In [ ]:
# Plot training and validation loss and accuracy over epochs
# Visualize model performance over epochs to detect learning progress, overfitting, or underfitting
def plot_history(history):
  fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,6))
  ax1.plot(history.history['loss'], label = 'loss')
  ax1.plot(history.history['val_loss'], label = 'val_loss')
  ax1.set_xlabel('Epoch')
  ax1.set_ylabel('Binary crossentropy')
  ax1.grid(True)
  ax2.plot(history.history['accuracy'], label = 'accuracy')
  ax2.plot(history.history['val_accuracy'], label = 'val_accuracy')
  ax2.set_xlabel('Epoch')
  ax2.set_ylabel('Accuracy')
  ax2.grid(True)
  plt.show()

In [ ]:
# Build, compile, and train a neural network model with configurable parameters
def train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs):
  nn_model =  tf.keras.Sequential([
      tf.keras.layers.Dense(num_nodes,activation = 'relu', input_shape = (10,)),
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(num_nodes,activation = 'relu'),
      tf.keras.layers.Dropout(dropout_prob),
      tf.keras.layers.Dense(1,activation = 'sigmoid')
  ])
  nn_model.compile(optimizer = tf.keras.optimizers.Adam(lr), loss = 'binary_crossentropy', metrics = ['accuracy'])
  history = nn_model.fit(X_train, y_train, epochs = epochs, batch_size = batch_size, validation_split = 0.2, verbose = 0)
  return nn_model, history

In [ ]:
# Perform grid search over hyperparameters to find the model with lowest validation loss
least_val_loss = float('inf')
least_loss_model = None
epochs = 100
for num_nodes in[16, 32,64]:
  for dropout_prob in [0,0.2]:
    for lr in [0.01, 0.005, 0.001]:
      for batch_size in [32, 64, 128]:
        print(f"{num_nodes} nodes, dropout {dropout_prob}, lr {lr}, batch_size {batch_size}")
        model, history = train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs)
        plot_history(history)
        val_loss_metrics = model.evaluate(X_valid, y_valid, verbose=0)
        val_loss = val_loss_metrics[0] # Get the loss value from the list of metrics
        if val_loss < least_val_loss:
          least_val_loss = val_loss
          least_loss_model = model

In [ ]:
y_pred = least_loss_model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int).reshape(-1,)

In [ ]:
print(classification_report(y_test, y_pred))
